# Stock Predictor MVP

**DISCLAIMER: This is NOT financial advice. This is for educational purposes only.**

This notebook predicts 3-day forward returns for S&P 500 stocks using machine learning.

## 1. Install Dependencies

In [ ]:
!pip install pandas numpy yfinance scikit-learn scipy joblib lightgbm lxml -q

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import logging
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.stats import spearmanr
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Try LightGBM, fall back to sklearn
try:
    from lightgbm import LGBMRegressor
    USE_LIGHTGBM = True
    print("Using LightGBM")
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor
    USE_LIGHTGBM = False
    print("Using sklearn HistGradientBoostingRegressor")

logging.basicConfig(level=logging.INFO)
print("Imports complete!")

## 3. Configuration

In [ ]:
# Configuration - adjust these as needed
MAX_TICKERS = 50       # Number of stocks to analyze (set to None for all 500)
YEARS_OF_DATA = 3      # Years of historical data
FORWARD_DAYS = 3       # Prediction horizon (trading days)
N_CV_SPLITS = 5        # Cross-validation splits

## 4. Get S&P 500 Tickers

In [ ]:
def get_sp500_tickers(max_tickers=None):
    """Fetch S&P 500 tickers from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    
    try:
        tables = pd.read_html(url)
        df = tables[0]
        
        if 'Symbol' in df.columns:
            tickers = df['Symbol'].tolist()
        else:
            tickers = df.iloc[:, 0].tolist()
        
        # Clean tickers
        cleaned = [t.replace(".", "-").strip() for t in tickers if isinstance(t, str)]
        
        if max_tickers:
            return cleaned[:max_tickers]
        return cleaned
    
    except Exception as e:
        print(f"Failed to fetch tickers: {e}")
        # Fallback list
        fallback = ["AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "TSLA", 
                    "UNH", "XOM", "JNJ", "JPM", "V", "PG", "MA", "HD", "CVX",
                    "MRK", "LLY", "ABBV", "PEP", "KO", "COST", "AVGO", "WMT"]
        if max_tickers:
            return fallback[:max_tickers]
        return fallback

tickers = get_sp500_tickers(MAX_TICKERS)
print(f"Got {len(tickers)} tickers")
print(f"First 10: {tickers[:10]}")

## 5. Fetch Price Data

In [ ]:
def fetch_ticker_data(ticker, start_date, end_date):
    """Fetch OHLCV data for a single ticker."""
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(start=start_date, end=end_date, auto_adjust=True)
        
        if df.empty:
            return None
        
        required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        if not all(col in df.columns for col in required_cols):
            return None
        
        df = df[required_cols].copy()
        if df.index.tz is not None:
            df.index = df.index.tz_localize(None)
        
        return df.dropna()
    except:
        return None

def fetch_all_prices(tickers, years=3, max_workers=10):
    """Fetch price data for all tickers in parallel."""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=years * 365)
    
    results = {}
    failed = []
    
    print(f"Fetching {len(tickers)} tickers...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_ticker = {
            executor.submit(fetch_ticker_data, ticker, start_date, end_date): ticker
            for ticker in tickers
        }
        
        for i, future in enumerate(as_completed(future_to_ticker)):
            ticker = future_to_ticker[future]
            try:
                df = future.result()
                if df is not None and len(df) > 0:
                    results[ticker] = df
                else:
                    failed.append(ticker)
            except:
                failed.append(ticker)
            
            if (i + 1) % 20 == 0:
                print(f"  Progress: {i + 1}/{len(tickers)}")
    
    print(f"\nLoaded {len(results)} tickers, {len(failed)} failed")
    return results

# Fetch all price data
print("Fetching stock prices (this may take a few minutes)...")
prices = fetch_all_prices(tickers, years=YEARS_OF_DATA)

# Fetch SPY for market features
print("\nFetching SPY data...")
spy_df = fetch_ticker_data("SPY", 
                           datetime.now() - timedelta(days=YEARS_OF_DATA * 365),
                           datetime.now())
print(f"SPY data: {len(spy_df)} rows")

## 6. Feature Engineering

In [ ]:
def compute_rsi(prices, period=14):
    """Compute RSI indicator."""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    
    avg_gain = gain.ewm(span=period, adjust=False).mean()
    avg_loss = loss.ewm(span=period, adjust=False).mean()
    
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_market_features(spy_df, target_index):
    """Compute market (SPY) features."""
    close = spy_df['Close']
    log_close = np.log(close)
    daily_ret = log_close.diff(1)
    
    mkt = pd.DataFrame(index=spy_df.index)
    mkt['ret_1d'] = log_close.diff(1)
    mkt['ret_5d'] = log_close.diff(5)
    mkt['ret_20d'] = log_close.diff(20)
    mkt['vol_20d'] = daily_ret.rolling(20).std()
    mkt['mom_20d'] = daily_ret.rolling(20).mean()
    mkt['rsi_14'] = compute_rsi(close, 14)
    
    return mkt.reindex(target_index, method='ffill')

def compute_features(df, ticker, spy_df=None, include_target=True, forward_days=3):
    """Compute all features for a single ticker."""
    if len(df) < 250:
        return pd.DataFrame()
    
    df = df.copy().sort_index()
    
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    features = pd.DataFrame(index=df.index)
    features['ticker'] = ticker
    
    close = df['Close']
    log_close = np.log(close)
    daily_ret = log_close.diff(1)
    
    # Return features
    features['ret_1d'] = log_close.diff(1)
    features['ret_3d'] = log_close.diff(3)
    features['ret_5d'] = log_close.diff(5)
    features['ret_10d'] = log_close.diff(10)
    features['ret_20d'] = log_close.diff(20)
    
    # Volatility features
    features['vol_5d'] = daily_ret.rolling(5).std()
    features['vol_10d'] = daily_ret.rolling(10).std()
    features['vol_20d'] = daily_ret.rolling(20).std()
    
    # Momentum features
    features['mom_5d'] = daily_ret.rolling(5).mean()
    features['mom_10d'] = daily_ret.rolling(10).mean()
    features['mom_20d'] = daily_ret.rolling(20).mean()
    
    # RSI
    features['rsi_14'] = compute_rsi(close, 14)
    
    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    
    features['macd_line'] = macd_line / close
    features['macd_signal'] = signal_line / close
    features['macd_hist'] = (macd_line - signal_line) / close
    
    # Moving average distances
    features['dist_ma20'] = (close / close.rolling(20).mean()) - 1
    features['dist_ma50'] = (close / close.rolling(50).mean()) - 1
    features['dist_ma200'] = (close / close.rolling(200).mean()) - 1
    
    # Volume features
    volume = df['Volume'].replace(0, np.nan)
    features['vol_pct_change'] = volume.pct_change()
    vol_mean = volume.rolling(20).mean()
    vol_std = volume.rolling(20).std()
    features['vol_zscore_20'] = (volume - vol_mean) / vol_std
    
    # Price range features
    high, low = df['High'], df['Low']
    tr = pd.concat([high - low, abs(high - close.shift(1)), abs(low - close.shift(1))], axis=1).max(axis=1)
    features['atr_14'] = tr.rolling(14).mean() / close
    features['daily_range'] = (high - low) / close
    
    # Market features
    if spy_df is not None and len(spy_df) > 0:
        mkt_features = compute_market_features(spy_df, features.index)
        for col in mkt_features.columns:
            features[f'mkt_{col}'] = mkt_features[col]
    
    # Target variable
    if include_target:
        future_close = close.shift(-forward_days)
        features['target'] = (future_close / close) - 1
    
    return features

# Get feature columns (excluding ticker and target)
sample_features = compute_features(list(prices.values())[0], "SAMPLE", spy_df)
FEATURE_COLUMNS = [col for col in sample_features.columns if col not in ['ticker', 'target']]
print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Features: {FEATURE_COLUMNS}")

## 7. Create Training Dataset

In [ ]:
def create_training_dataset(prices, spy_df, forward_days=3):
    """Create pooled training dataset from all tickers."""
    all_features = []
    
    for ticker, df in prices.items():
        try:
            features = compute_features(df, ticker, spy_df, include_target=True, forward_days=forward_days)
            if len(features) > 0:
                all_features.append(features)
        except Exception as e:
            pass
    
    if not all_features:
        return pd.DataFrame()
    
    combined = pd.concat(all_features, axis=0)
    combined = combined.dropna(subset=['target'])
    combined = combined.dropna(subset=FEATURE_COLUMNS)
    
    print(f"Dataset: {len(combined)} samples from {len(prices)} tickers")
    return combined

print("Creating training dataset...")
dataset = create_training_dataset(prices, spy_df, FORWARD_DAYS)
print(f"\nTarget statistics:")
print(dataset['target'].describe())

## 8. Train Model

In [ ]:
def get_model():
    """Get the ML model."""
    if USE_LIGHTGBM:
        return LGBMRegressor(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            verbose=-1,
            n_jobs=-1
        )
    else:
        return HistGradientBoostingRegressor(
            max_iter=200,
            max_depth=6,
            learning_rate=0.05,
            min_samples_leaf=20,
            l2_regularization=0.1,
            random_state=42
        )

def train_model(dataset, feature_columns, n_splits=5):
    """Train model with time-series cross-validation."""
    dataset = dataset.sort_index()
    
    X = dataset[feature_columns].values
    y = dataset['target'].values
    
    print(f"Training on {len(dataset)} samples with {len(feature_columns)} features")
    print("\nCross-validation results:")
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    cv_metrics = {'mse': [], 'mae': [], 'spearman': []}
    all_residuals = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', get_model())
        ])
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)
        
        mse = mean_squared_error(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        spearman = spearmanr(y_val, y_pred)
        
        cv_metrics['mse'].append(mse)
        cv_metrics['mae'].append(mae)
        cv_metrics['spearman'].append(spearman.statistic)
        
        all_residuals.extend(y_val - y_pred)
        
        print(f"  Fold {fold+1}: MSE={mse:.6f}, MAE={mae:.6f}, Spearman={spearman.statistic:.4f}")
    
    # Final model on all data
    print("\nTraining final model on all data...")
    final_model = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', get_model())
    ])
    final_model.fit(X, y)
    
    residual_std = np.std(all_residuals)
    
    print(f"\n" + "="*50)
    print(f"CV Results:")
    print(f"  MSE: {np.mean(cv_metrics['mse']):.6f} +/- {np.std(cv_metrics['mse']):.6f}")
    print(f"  Spearman: {np.mean(cv_metrics['spearman']):.4f} +/- {np.std(cv_metrics['spearman']):.4f}")
    print(f"="*50)
    
    return final_model, residual_std

model, residual_std = train_model(dataset, FEATURE_COLUMNS, N_CV_SPLITS)

## 9. Generate Predictions

In [ ]:
def create_prediction_features(prices, spy_df):
    """Create features for prediction (latest date only)."""
    latest_features = []
    
    for ticker, df in prices.items():
        try:
            features = compute_features(df, ticker, spy_df, include_target=False)
            if len(features) > 0:
                valid = features.dropna(subset=FEATURE_COLUMNS)
                if len(valid) > 0:
                    latest_features.append(valid.iloc[[-1]].copy())
        except:
            pass
    
    return pd.concat(latest_features, axis=0) if latest_features else pd.DataFrame()

def predict(model, features, residual_std):
    """Generate predictions with confidence intervals."""
    X = features[FEATURE_COLUMNS].values
    predictions = model.predict(X)
    
    result = pd.DataFrame(index=features.index)
    result['ticker'] = features['ticker'].values
    result['predicted_return'] = predictions
    result['pred_lower'] = predictions - 1.96 * residual_std
    result['pred_upper'] = predictions + 1.96 * residual_std
    
    return result

print("Generating predictions for latest data...")
pred_features = create_prediction_features(prices, spy_df)
predictions = predict(model, pred_features, residual_std)
predictions = predictions.sort_values('predicted_return', ascending=False)

print(f"\nGenerated predictions for {len(predictions)} stocks")

## 10. Results

In [ ]:
print("=" * 60)
print(f"  TOP 10 PREDICTED RETURNS (Next {FORWARD_DAYS} Trading Days)")
print("=" * 60)
print(f"{'Rank':<6} {'Ticker':<8} {'Predicted':<12} {'95% CI'}")
print("-" * 60)

for i, (_, row) in enumerate(predictions.head(10).iterrows()):
    pred = row['predicted_return'] * 100
    lower = row['pred_lower'] * 100
    upper = row['pred_upper'] * 100
    print(f"{i+1:<6} {row['ticker']:<8} {pred:+.2f}%       [{lower:+.2f}%, {upper:+.2f}%]")

print("\n" + "=" * 60)
print(f"  BOTTOM 10 PREDICTED RETURNS (Next {FORWARD_DAYS} Trading Days)")
print("=" * 60)
print(f"{'Rank':<6} {'Ticker':<8} {'Predicted':<12} {'95% CI'}")
print("-" * 60)

for i, (_, row) in enumerate(predictions.tail(10).iloc[::-1].iterrows()):
    pred = row['predicted_return'] * 100
    lower = row['pred_lower'] * 100
    upper = row['pred_upper'] * 100
    print(f"{i+1:<6} {row['ticker']:<8} {pred:+.2f}%       [{lower:+.2f}%, {upper:+.2f}%]")

In [ ]:
# Full predictions table
print("\nFull Predictions Table:")
display_df = predictions.copy()
display_df['predicted_return'] = (display_df['predicted_return'] * 100).round(3).astype(str) + '%'
display_df['pred_lower'] = (display_df['pred_lower'] * 100).round(3).astype(str) + '%'
display_df['pred_upper'] = (display_df['pred_upper'] * 100).round(3).astype(str) + '%'
display_df

---

## ⚠️ DISCLAIMER

**This is NOT financial advice!**

- Past performance does not guarantee future results
- Stock predictions are inherently uncertain
- This model is for educational purposes only
- Always do your own research before investing
- Never invest money you can't afford to lose